In [1]:
import os
import sys
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from sleep_analysis_functions import sleep_stage_aswti
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import *

In [2]:
dir_input = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step6' # compute ahi versions etc
files = os.listdir(dir_input)
files.sort()
file = files[59]

In [3]:
data, hdr, fs = read_in_routine(os.path.join(dir_input, file))

In [4]:
def compute_spo2_clean(data):
    
    data['spo2_clean'] = data['spo2'].copy()
    
    spo2_badqualityth = data.spo2.median() - 30
    spo2_badq = np.where(data.spo2 < spo2_badqualityth)[0]

    if len(spo2_badq) > 0:

        spo2_badq_start = np.concatenate([[spo2_badq[0]], spo2_badq[1:][np.diff(spo2_badq) > 1]])
        spo2_badq_end = np.concatenate([[spo2_badq[-1]], spo2_badq[::-1][:-1][np.diff(spo2_badq[::-1]) < -1]])[::-1]

        bad_quality = (data.spo2 < spo2_badqualityth).astype(int).reset_index()
        bad_quality = bad_quality.rolling(20*fs, min_periods=0).max()
        bad_quality = (bad_quality[::-1].rolling(20*fs, min_periods=0).max())[::-1]
        bad_quality.index = data.index

        data.loc[bad_quality.values.astype(bool).flatten(), 'spo2_clean'] = np.nan

        bad_quality_prior = bad_quality.reset_index()[::-1].rolling(5*60*fs, min_periods=0).max()[::-1]
        bad_quality_post = bad_quality.reset_index().rolling(5*60*fs, min_periods=0).max()

        bad_quality_prior_10min = bad_quality.reset_index()[::-1].rolling(10*60*fs, min_periods=0).max()[::-1]
        bad_quality_post_10min = bad_quality.reset_index().rolling(10*60*fs, min_periods=0).max()

        bad_q_area = bad_quality_prior['spo2'].values.astype(bool) & bad_quality_post['spo2'].values.astype(bool)
        bad_q_area = pd.Series(bad_q_area)
        bad_q_area.index = data.index
        data.loc[bad_q_area.values.astype(bool).flatten(), 'spo2_clean'] = np.nan
        # once a bad quality has been detected, remove all data under 90 in the area, if still included:
        data.loc[bad_quality_prior_10min.values.astype(bool).flatten() & (data.spo2_clean <= 90), 'spo2_clean'] = np.nan
        data.loc[bad_quality_post_10min.values.astype(bool).flatten() & (data.spo2_clean <= 90), 'spo2_clean'] = np.nan
        
    return data

In [10]:
def compute_spo2_perc_below_90(data):
    
    if not 'spo2_clean' in data.columns:
        return data
    
    spo2_clean = data.spo2_clean.dropna().values
    spo2_perc_below_90 = len(spo2_clean[spo2_clean < 90]) / len(spo2_clean)
    
    return spo2_perc_below_90

In [11]:
def bridge_small_spo2_drops(data):
    
    drop_magnitude = 3
    max_gap = 10
    
    data_drop = data.spo2_clean_drop_45s.dropna()
    
    start = data_drop[data_drop >= drop_magnitude].diff() == 1
    end = data_drop[data_drop >= drop_magnitude].diff() == -1
    if data_drop[data_drop >= drop_magnitude].diff().iloc[0] == 1:
        start.iloc[0] = True
    if data_drop[data_drop >= drop_magnitude].diff().iloc[-1] == 1:
        end.iloc[-1] = True
    start_idx = np.where(start)[0]
    end_idx = np.where(end)[0]
    
    end_indices_close_to_next_start = np.where(start_idx[1:] - end_idx[:-1] < max_gap)[0]
    for idx_tmp in end_indices_close_to_next_start:
        loc_to_change = data_drop[data_drop >= drop_magnitude].iloc[end_idx[idx_tmp]: start_idx[idx_tmp+1]].index
        data.loc[loc_to_change, 'spo2_clean_drop_45s'] = drop_magnitude
    
    return data

In [12]:
def hypoxia_drops(data, drop_magnitude=3, max_gap=10):
    
    if not 'spo2_clean' in data.columns:
        return data

    data['spo2_clean_drop_45s'] = np.nan
    data.loc[data.index[::fs], 'spo2_clean_drop_45s']= data['spo2_clean'][::fs].astype('float32').rolling(45, min_periods=5).apply(lambda x:  x.max() - x[-1],raw=False)

    # get the start and end_idx of the drops
    data_drop = data.spo2_clean_drop_45s.dropna() # supposed to be in 1-second resolution here.
    start = (data_drop >= drop_magnitude).astype(int).diff() == 1
    end = (data_drop >= drop_magnitude).astype(int).diff() == -1
    if len(start[start==True]) == len(end[end==True]) + 1:
        end.iloc[-1] = 1

    start_idx = np.where(start)[0]
    end_idx = np.where(end)[0]

    # for end-start times of drops that are close together, bridge the drop value so they are connected
    end_indices_close_to_next_start = np.where(start_idx[1:] - end_idx[:-1] < max_gap)[0]
    for idx_tmp in end_indices_close_to_next_start:
        loc_to_change = data_drop.iloc[end_idx[idx_tmp]: start_idx[idx_tmp+1]].index
        data.loc[loc_to_change, 'spo2_clean_drop_45s'] = drop_magnitude

    # after connection, let's get updated start and end_idx
    data_drop = data.spo2_clean_drop_45s.dropna()
    start = (data_drop >= drop_magnitude).astype(int).diff() == 1
    end = (data_drop >= drop_magnitude).astype(int).diff() == -1
    if len(start[start==True]) == len(end[end==True]) + 1:
        end.iloc[-1] = 1

    start_idx = np.where(start)[0]
    end_idx = np.where(end)[0]

    min_drop_duration = 10
    hypoxia_drop_short = end_idx - start_idx < min_drop_duration
    hypoxia_drop_long = end_idx - start_idx >= min_drop_duration

    # start_loc = data_drop[start_idx[hypoxia_drop_short]].index
    # end_loc = data_drop[end_idx[hypoxia_drop_short]].index
    
    no_hypoxia_short = len(start_idx[hypoxia_drop_short])
    no_hypoxia_long = len(start_idx[hypoxia_drop_long])
        
    return data, no_hypoxia_short, no_hypoxia_long

In [13]:
data = compute_spo2_clean(data)
spo2_perc_below_90 = compute_spo2_perc_below_90(data)
data, no_hypoxia_short, no_hypoxia_long = hypoxia_drops(data)

print(f'Fraction of SpO2 < 90: {spo2_perc_below_90}')
print(f'Number of short desats: {no_hypoxia_short}')
print(f'Number of long desats: {no_hypoxia_long}')

Fraction of SpO2 < 90: 0.005189811965621557
Number of short desats: 164
Number of long desats: 93


In [14]:
if 1:
    for file in files:
        data, hdr, fs = read_in_routine(os.path.join(dir_input, file))
        print(file)
        data = compute_spo2_clean(data)
        spo2_perc_below_90 = compute_spo2_perc_below_90(data)
        data, no_hypoxia_short, no_hypoxia_long = hypoxia_drops(data)

        print(f'Fraction of SpO2 < 90: {spo2_perc_below_90}')
        print(f'Number of short desats: {no_hypoxia_short}')
        print(f'Number of long desats: {no_hypoxia_long}')
        print('')

icusleep_001.h5
Fraction of SpO2 < 90: 0.01927241677706899
Number of short desats: 821
Number of long desats: 570

icusleep_003.h5
Fraction of SpO2 < 90: 0.014680937711341543
Number of short desats: 322
Number of long desats: 374

icusleep_004.h5
Fraction of SpO2 < 90: 0.004096313857440503
Number of short desats: 313
Number of long desats: 241

icusleep_006.h5
Fraction of SpO2 < 90: 0.003563450029849249
Number of short desats: 635
Number of long desats: 1235

icusleep_007.h5
Fraction of SpO2 < 90: 0.008305055405717021
Number of short desats: 1257
Number of long desats: 1097

icusleep_011.h5
Fraction of SpO2 < 90: 0.00563223930666212
Number of short desats: 132
Number of long desats: 107

icusleep_012.h5


KeyboardInterrupt: 

In [154]:
if data_drop[data_drop >= drop_magnitude].diff().iloc[0] == 1:
    start.iloc[0] = True
if data_drop[data_drop >= drop_magnitude].diff().iloc[-1] == 1:
    end.iloc[-1] = True
start_idx = np.where(start)[0]
end_idx = np.where(end)[0]

end_indices_close_to_next_start = np.where(start_idx[1:] - end_idx[:-1] < max_gap)[0]
for idx_tmp in end_indices_close_to_next_start:
    loc_to_change = data_drop[data_drop >= drop_magnitude].iloc[end_idx[idx_tmp]: start_idx[idx_tmp+1]].index
    data.loc[loc_to_change, 'spo2_clean_drop_45s'] = drop_magnitude


In [174]:
plt.figure(figsize=(6,3))
# plt.plot(data.spo2)
plt.plot(data.spo2_clean)
plt.plot(data.spo2_clean_drop_45s.dropna())
plt.scatter(data_drop.index[start_idx[hypoxia_drop_short]], [90]*len([start_idx[hypoxia_drop_short]][0]), s=4, color='red')
plt.scatter(data_drop.index[start_idx[hypoxia_drop_long]], [92]*len([start_idx[hypoxia_drop_long]][0]), s=4, color='green')

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …